# City Workflow — TerrainSession

End-to-end city workflow using the session SDK:

1. Start server & select a city-scale region
2. Fetch DEM + city data
3. Composite city raster (buildings, roads, waterways, walls)
4. Rasterize city (merged height deltas)
5. Visualize layers
6. Export city 3MF
7. Full pipeline with `run_all()`

Docs: [sdk-workflow.md](../docs/sdk-workflow.md) · [api.md](../docs/api.md)

In [ ]:
%matplotlib inline

In [ ]:
import sys, os
os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
sys.path.insert(0, os.path.abspath(".."))

from app.session.terrain_session import TerrainSession

## 1. Start Server & Select Region

In [ ]:
s = TerrainSession(port=9000)
s.start(visible=True)

In [ ]:
# List available regions — pick one with city-scale bbox
s.regions()

In [ ]:
# Select a small urban region (StatenIsland is ~20 km across)
s.select("StatenIsland")

# Configure settings for city work
s.settings["dem"]["dim"] = 600
s.settings["projection"]["projection"] = "none"  # or "sinusoidal"

# City-specific settings
s.settings["city"]["building_scale"] = 0.5
s.settings["city"]["road_depression_m"] = 3.0
s.settings["city"]["water_depression_m"] = 5.0

s.settings_table()

## 2. Fetch DEM

In [ ]:
s.fetch_dem()
s.show_dem()

## 3. Fetch City Data (OSM buildings, roads, waterways)

In [ ]:
# Check if city data is already cached
cached = s.check_city_cache()
print(f"Cached: {cached}")

In [ ]:
# Fetch OSM data (uses cache if available)
s.fetch_cities()

# Quick summary of what was fetched
if s.city_data:
    for layer in ["buildings", "roads", "waterways"]:
        feats = s.city_data.get(layer, {}).get("features", [])
        print(f"  {layer}: {len(feats)} features")

## 4. Composite City Raster (per-layer arrays)

In [ ]:
# Rasterize from disk cache — returns separate arrays for buildings, roads,
# waterways, and walls.  Projection is applied server-side.
s.composite_city_raster()
s.show_city()

## 5. Rasterize City (merged height-delta grid)

In [ ]:
# Alternative: single merged height map compatible with merge_dem()
s.rasterize_city()

## 6. Export City 3MF

In [ ]:
# Export terrain + extruded buildings as 3MF
data_3mf = s.export_city_3mf()

# Save to disk
out_path = os.path.join("output", "city_export.3mf")
os.makedirs("output", exist_ok=True)
with open(out_path, "wb") as f:
    f.write(data_3mf)
print(f"Saved to {out_path}")

## 7. Full Pipeline

For the standard terrain-only pipeline, use `run_all()`.
For terrain + city, chain the steps manually:

In [ ]:
# Terrain + city pipeline (chain)
s.select("StatenIsland")
s.fetch_dem()
s.fetch_cities()
s.composite_city_raster()
s.show_dem()
s.show_city()

## 8. Cleanup

In [ ]:
s.stop()